# Lab 2 - Your First Python Session

**Estimated time:** 15 min  ·  **Estimated cost:** a few cents

This is a **Jupyter Notebook**; run it top to bottom in **Udemy Labs** (nothing to install) or on your own machine. Read the note above each cell, then run the cell with Shift+Enter.

## Goal
Recreate the Lab 1 coding-assistant agent in about a dozen lines of Python and stream a turn. By the end you will have created an agent, a cloud environment, and a session entirely from code, and watched the agent write and run `fib.py` live. This is the bridge between the no-code Console and the API you will use for the rest of the course.

## Prerequisites
- The shared uv environment. Run `./setup_uv.sh` from `labs/code`, then select the `Managed Agents Labs (.venv)` kernel.
- An API key you can paste into the setup cell below. The notebook stores it as `ANTHROPIC_API_KEY` for this kernel session only.
- Lab 1 finished, so you know what the agent should do.

> The Managed Agents endpoints need a beta header. You do not set it by hand: every call under `client.beta.*` adds it for you. That is why the code uses `client.beta.agents`, `client.beta.environments`, and `client.beta.sessions`.

In [ ]:
# Verify that this notebook is using the uv-managed lab kernel.
import sys
from pathlib import Path

EXPECTED_KERNEL = "Managed Agents Labs (.venv)"
python_exe = Path(sys.executable)
python_prefix = Path(sys.prefix)

if ".venv" not in python_exe.parts and ".venv" not in python_prefix.parts:
    raise RuntimeError(
        f"Select the Jupyter kernel {EXPECTED_KERNEL!r} before running this notebook. "
        "From labs/code, run ./setup_uv.sh once to create and register it."
    )

print("Using uv environment:", python_prefix)

In [ ]:
import getpass
import os

if not os.environ.get("ANTHROPIC_API_KEY"):
    os.environ["ANTHROPIC_API_KEY"] = getpass.getpass("Enter your Anthropic API key: ")

print("ANTHROPIC_API_KEY configured for this notebook session.")

In [ ]:
import os, sys
from anthropic import Anthropic

# Udemy Labs note: the previous cell configures ANTHROPIC_API_KEY for this session.
assert os.environ.get("ANTHROPIC_API_KEY"), "Set ANTHROPIC_API_KEY first."

BETAS = ["managed-agents-2026-04-01"]
MODEL = os.environ.get("MODEL", "claude-haiku-4-5-20251001")  # course default; swap as models update
client = Anthropic()
print("SDK ready, model:", MODEL)

### Step 1 - Create the agent
The model, system prompt, and toolset live on the **agent**, not the session. Use the course default model and the prebuilt agent toolset.

> Both the model id (`claude-haiku-4-5-20251001`) and the toolset id (`agent_toolset_20260401`) are current values. They are versioned on purpose, so a newer model or a newer dated toolset may exist by the time you watch this. The flow does not change.

In [ ]:
agent = client.beta.agents.create(
    name="Coding Assistant",
    model=MODEL,
    system="You are a helpful coding assistant. Write clean code and verify it runs.",
    tools=[{"type": "agent_toolset_20260401"}],  # date-stamped toolset; may update
    betas=BETAS,
)
print("agent.id   =", agent.id)

### Step 2 - Create the cloud environment
The environment is a template for the container the agent's tools run in. A cloud environment with unrestricted networking is the simplest choice for a lab.

In [ ]:
env = client.beta.environments.create(
    name="lab02-env",
    config={"type": "cloud", "networking": {"type": "unrestricted"}},
    betas=BETAS,
)
print("env.id     =", env.id)

### Step 3 - Create the session
The session ties the agent to the environment. Pin it to the exact agent version you just created.

In [ ]:
session = client.beta.sessions.create(
    agent={"type": "agent", "id": agent.id, "version": agent.version},
    environment_id=env.id,
    title="Lab 2 first session",
    betas=BETAS,
)
print("session.id =", session.id)

### Step 4 - Send a message and stream events
Open the stream **first**, then send the user message, so you do not miss any early events. Print the agent's text as it arrives, print a line for each tool use, and stop when the session goes idle.

In [ ]:
with client.beta.sessions.events.stream(session.id, betas=BETAS) as stream:
    client.beta.sessions.events.send(
        session.id,
        events=[{
            "type": "user.message",
            "content": [{
                "type": "text",
                "text": "Create /workspace/fib.py that prints the first 20 "
                        "Fibonacci numbers. Then run it.",
            }],
        }],
        betas=BETAS,
    )

    for event in stream:
        if event.type == "agent.message":
            for block in event.content:
                if block.type == "text":
                    sys.stdout.write(block.text); sys.stdout.flush()
        elif event.type == "agent.tool_use":
            print(f"\n[tool: {event.name}]")
        elif event.type == "session.status_idle":
            print("\n--- session idle ---")
            break

### Cost estimate
Re-fetch the session id(s), print one list-price estimate per session, then print the total across all session ids. The estimate uses cumulative token usage plus Managed Agents active runtime; treat it as a course meter, not an invoice.


In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "shared"))
from cost_meter import print_sessions_cost

print_sessions_cost(client, [session.id], MODEL, betas=BETAS)


## Expected output
You should see, in order:

- Three printed IDs: `agent.id`, `env.id`, `session.id`.
- Streamed `[tool: ...]` lines as the agent works - typically `write` (creating `fib.py`), then `bash` (running it), and maybe `read`.
- The agent's chat text explaining what it created and the output it observed (the first 20 Fibonacci numbers).
- A final `--- session idle ---` line.

The agent has now written `/workspace/fib.py` and executed it inside the cloud container, exactly like Lab 1 but driven from code.

## Troubleshooting
- **`AuthenticationError` / 401** - your key is missing or wrong. Confirm `ANTHROPIC_API_KEY` prints `sk-ant-...` and re-set it if not (see Chapter 2).
- **400 "missing beta header"** - you called a non-beta path. Managed Agents lives under `client.beta.*`. Use `client.beta.agents`, `client.beta.environments`, `client.beta.sessions` - the beta header is added for you there.
- **`ModuleNotFoundError: No module named 'anthropic'`** - the SDK is not installed in this kernel. Select the `Managed Agents Labs (.venv)` kernel, then restart the kernel.
- **`AttributeError` on `client.beta.agents`** - your SDK is too old for Managed Agents. run `./setup_uv.sh` from `labs/code`, then restart the kernel.
- **Wrong interpreter / venv** - outside Udemy Labs, make sure the notebook kernel is the same Python you installed `anthropic` into.

## Bonus (optional): drive this lab with Claude Code

Not required - the notebook above is the whole lab. If you prefer to **generate** the script with agentic engineering instead of running the cells, open this folder in Claude Code and paste these prompts in order:

1. > Using the Anthropic Python SDK, write a minimal script that creates a Managed Agent named "Coding Assistant" with model `claude-haiku-4-5-20251001` and the `agent_toolset_20260401` toolset, then creates a cloud environment with unrestricted networking, then a session that references the agent. Use the `client.beta.*` namespace so the beta header is added automatically.

2. > Now add a streamed turn: open the session event stream, send a user message asking the agent to create `/workspace/fib.py` that prints the first 20 Fibonacci numbers and run it, then print the agent's text and each tool use, stopping when the session goes idle.

3. > Run the script and show me the output.

Compare what Claude Code writes to the cells above.

## Stretch
- **Ask a follow-up in the same session.** After the session goes idle, send another `user.message` (for example, "Now make it print the first 30 numbers and rerun it") and stream again. The session keeps the container and its files, so the agent edits the existing `fib.py`.
- **Print a running token count.** The `span.model_request_end` event carries `model_usage`. Add a branch that accumulates `input_tokens` and `output_tokens` and prints a running total as the turn progresses:

  ```python
  total_in = total_out = 0
  # inside the for-loop:
  elif event.type == "span.model_request_end":
      u = event.model_usage
      total_in += u.input_tokens
      total_out += u.output_tokens
      print(f"\n[tokens so far: in={total_in} out={total_out}]")
  ```